# $D^+D^-$ background in $K_L\pi^0$
## Calculate the background yield of the $D^+D^-$ background in the $K_L\pi^0$ tag

### Include library for handling uncertainties
#### [Here is the ```uncertainties-cpp``` library on GitHub](https://github.com/Gattocrucco/uncertainties-cpp)

In [1]:
gInterpreter->AddIncludePath("/data/lhcb/users/tat/uncertainties-cpp");

In [2]:
#include<uncertainties/impl.hpp>
#include<uncertainties/ureal.hpp>
#include<uncertainties/io.hpp>
#include<uncertainties/math.hpp>
#include<uncertainties/stat.hpp>

### Load utility functions

In [3]:
gROOT->ProcessLine(".L ../UtilityFunctions.C");

### Number of bins

In [4]:
const int NumberBins = 4;

### Get fitted values of $c_i$, $s_i$ and $K_i$

In [5]:
const std::string cisiFilename("BiasCorrectedResults.txt");
auto cisiFitted = ParseParameters(cisiFilename);
std::map<int, uncertainties::udouble> ci, si, Ki;
for(int Bin = 1; Bin <= NumberBins; Bin++) {
    const std::string BinName = std::to_string(Bin);
    ci[Bin] = uncertainties::udouble(cisiFitted["c" + BinName], cisiFitted["c" + BinName + "_err"]);
    si[Bin] = uncertainties::udouble(cisiFitted["s" + BinName], cisiFitted["s" + BinName + "_err"]);
    Ki[Bin] = uncertainties::udouble(cisiFitted["K" + BinName], cisiFitted["K" + BinName + "_err"]);
    Ki[-Bin] = uncertainties::udouble(cisiFitted["Kbar" + BinName], cisiFitted["Kbar" + BinName + "_err"]);
}

### Calculate ratio of branching fractions and cross sections

In [6]:
uncertainties::udouble CrossSection_D0D0(3.615, 0.039);
uncertainties::udouble CrossSection_DpDm(2.830, 0.028);

In [7]:
uncertainties::udouble KKpipi_BF(0.00247, 0.00011);
uncertainties::udouble KLpi0_BF(0.00959, 0.00032);
uncertainties::udouble KLpipi0_BF(0.0736, 0.0021);
uncertainties::udouble KKpipi0_BF(0.00662, 0.00032);

In [8]:
auto ProductionRatio = (CrossSection_DpDm/CrossSection_D0D0)*(KKpipi0_BF*KLpipi0_BF)/(KKpipi_BF*KLpi0_BF);
std::cout << ProductionRatio << "\n";

16.1 ± 1.3


### Calculate quantum correlation factor in each bin

In [9]:
std::map<int, uncertainties::udouble> QC_factors;
for(int Bin = 1; Bin <= NumberBins; Bin++) {
    QC_factors.insert({Bin, 1.0 - 2.0*(uncertainties::sqrt(Ki[Bin]*Ki[-Bin])/(Ki[Bin] + Ki[-Bin]))*ci[Bin]});
}

### Get total number of generated signal and background events

In [10]:
TChain GenSigEventsChain("TruthTuple");
std::string GenSigEventsFilename = "/data/bes3/tat/KKpipi_StrongPhase_Analysis_4Bins/TruthTuples/";
GenSigEventsFilename += "BinnedTruthTuples/KLpi0/KKpipi_vs_KLpi0_TruthTuple_Binned_Reweighted.root";
GenSigEventsChain.Add(GenSigEventsFilename.c_str());
auto GenSigYields = GetBinYields(&GenSigEventsChain, false, NumberBins, "ModelWeight_CPOdd");
double TotalGenSigYields = 0.0;
for(int Bin = 1; Bin <= NumberBins; Bin++) {
    TotalGenSigYields += GenSigYields[Bin];
}
// Need to account for KS veto, which were removed at generator level as well
TotalGenSigYields *= 800000.0/GenSigEventsChain.GetEntries();

In [11]:
double TotalGenBkgYields = 800000.0;

### Get total number of reconstructed events in each bin

In [12]:
TChain RecSigEventsChain("KLpi0DoubleTag");
std::string RecSigEventsFilename = "/data/bes3/tat/KKpipi_StrongPhase_Analysis_4Bins/Selection/SignalMC/";
RecSigEventsFilename += "DoubleTag/KLpi0/KKpipi_vs_KLpi0_Binned_SignalMC_Reweighted.root";
RecSigEventsChain.Add(RecSigEventsFilename.c_str());
auto RecSigYields = GetBinYields(&RecSigEventsChain, true, NumberBins, "ModelWeight_CPOdd");

In [13]:
TChain RecBkgEventsChain("KLpi0DoubleTag");
std::string RecBkgEventsFilename = "/data/bes3/tat/KKpipi_StrongPhase_Analysis_4Bins/Selection/";
RecBkgEventsFilename += "PeakingBackgrounds/DoubleTag/KLpi0/";
RecBkgEventsFilename += "KKpipi_vs_D+D-_to_KKpipi_vs_KLpi0_DoubleTag_SignalMC_Binned.root";
RecBkgEventsChain.Add(RecBkgEventsFilename.c_str());
auto RecBkgYields = GetBinYields(&RecBkgEventsChain, true, NumberBins, "");

### Calculate ratio of bin efficiencies

In [14]:
std::map<int, uncertainties::udouble> BinEfficiencyRatios;
for(int Bin = 1; Bin <= NumberBins; Bin++) {
    double p = RecSigYields[Bin]/TotalGenSigYields;
    double p_err = TMath::Sqrt(p*(1.0 - p)/TotalGenSigYields);
    uncertainties::udouble SigEff(p, p_err);
    p = RecBkgYields[Bin]/TotalGenBkgYields;
    p_err = TMath::Sqrt(p*(1.0 - p)/TotalGenBkgYields);
    uncertainties::udouble BkgEff(p, p_err);
    BinEfficiencyRatios.insert({Bin, BkgEff/SigEff});
}

### Print the final background-to-signal ratios and quantum correlation factors

In [15]:
std::string Label = "KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin";
std::vector<uncertainties::udouble> QCFactors;
for(int Bin = 1; Bin <= NumberBins; Bin++) {
    std::string NewLabel = Label + std::to_string(Bin) + "_QuantumCorrelationFactor";
    std::cout << NewLabel << " " << uncertainties::nom(QC_factors[Bin]) << "\n";
    QCFactors.push_back(QC_factors[Bin]);
    std::cout << NewLabel << "_err " << uncertainties::sdev(QC_factors[Bin]) << "\n";
}
std::string Filename = "PeakingBackground_DT_D+D-_to_KLpi0";
Filename += "_QuantumCorrelationFactors.root";
std::vector<double> FlatCovMatrix =
    uncertainties::cov_matrix<std::vector<double>>(QCFactors);
SaveCovMatrix(FlatCovMatrix, Filename);

KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin1_QuantumCorrelationFactor 1.29396
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin1_QuantumCorrelationFactor_err 0.109867
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin2_QuantumCorrelationFactor 0.293208
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin2_QuantumCorrelationFactor_err 0.050014
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin3_QuantumCorrelationFactor 0.195106
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin3_QuantumCorrelationFactor_err 0.0532083
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin4_QuantumCorrelationFactor 1.27805
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin4_QuantumCorrelationFactor_err 0.109586


In [16]:
std::vector<uncertainties::udouble> BkgToSigRatio;
for(int Bin = 1; Bin <= NumberBins; Bin++) {
    std::string NewLabel = Label + std::to_string(Bin) + "_BackgroundToSignalRatio";
    std::cout << NewLabel << " " << uncertainties::nom(BinEfficiencyRatios[Bin]*ProductionRatio) << "\n";
    BkgToSigRatio.push_back(BinEfficiencyRatios[Bin]*ProductionRatio);
    std::cout << NewLabel << "_err " << uncertainties::sdev(BinEfficiencyRatios[Bin]*ProductionRatio) << "\n";
}
std::string Filename = "PeakingBackground_DT_D+D-_to_KLpi0.root";
std::vector<double> FlatCovMatrix =
    uncertainties::cov_matrix<std::vector<double>>(BkgToSigRatio);
SaveCovMatrix(FlatCovMatrix, Filename);

KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin1_BackgroundToSignalRatio 0.0272802
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin1_BackgroundToSignalRatio_err 0.00634981
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin2_BackgroundToSignalRatio 0.0279709
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin2_BackgroundToSignalRatio_err 0.00651074
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin3_BackgroundToSignalRatio 0.0421174
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin3_BackgroundToSignalRatio_err 0.00866118
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin4_BackgroundToSignalRatio 0.0373533
KLpi0_PeakingBackground5_DoubleTag_CP_KKpipi_vs_KLpi0_SignalBin4_BackgroundToSignalRatio_err 0.0083577
